# BiRefNet Background Removal

This notebook loads the official BiRefNet model once and removes the background from an input image.

In [5]:
!pip install -q transformers torch torchvision pillow opencv-python kornia

In [2]:
from PIL import Image
import torch
from transformers import AutoModelForImageSegmentation
from torchvision import transforms

In [6]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "ZhengPeng7/BiRefNet"

model = AutoModelForImageSegmentation.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model.to(DEVICE)
model.eval()

print("Loaded on:", DEVICE)

model.safetensors: reconstructing file:   0%|          |  0.00B /  444MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/754 [00:00<?, ?it/s]

Loaded on: cpu


In [7]:
transform = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    ),
])

In [8]:
def remove_background(image_path, output_path="transparent.png"):
    image = Image.open(image_path).convert("RGB")
    original_size = image.size

    input_tensor = transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred = model(input_tensor)[-1].sigmoid().cpu()[0][0]

    pred = transforms.ToPILImage()(pred)
    pred = pred.resize(original_size)

    image = image.convert("RGBA")
    image.putalpha(pred)

    image.save(output_path)

    return image

In [9]:
def save_white_background(rgba_image, output_path="white_bg.png"):
    white = Image.new("RGBA", rgba_image.size, (255,255,255,255))
    white.paste(rgba_image, mask=rgba_image.split()[-1])
    white.convert("RGB").save(output_path)

In [ ]:
# IMAGE_PATH = r"C:\Users\VedantSonani\OneDrive - inkeysolutions.com\Desktop\Extension\images\Product 1\test.jpg"   # Change to your image path

from pathlib import Path
import io, os

# ==========================
# USER CONFIGURATION
# ==========================

IMAGE_REL_PATH = Path(r"images\Product 1\test.jpg")
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "images").exists() else Path.cwd().parent
INPUT_IMAGE = str((PROJECT_ROOT / IMAGE_REL_PATH).resolve()) # Input image path

rgba = remove_background(
    INPUT_IMAGE,
    "transparent.png"
)

save_white_background(
    rgba,
    "white_bg.png"
)

print("Done!")
print("Generated:")
print("- transparent.png")
print("- white_bg.png")

FileNotFoundError: [Errno 2] No such file or directory: 'https://drive.google.com/file/d/1cawkUvGacEt6W6WocGFTIBAjDqYA3CNs/view?usp=drive_link'

In [3]:
! pip install  opencv-python numpy

   ---------------------------------------- 0.0/44.0 MB ? eta -:--:--
   - -------------------------------------- 2.1/44.0 MB 13.6 MB/s eta 0:00:04
   ------ --------------------------------- 7.1/44.0 MB 20.1 MB/s eta 0:00:02
   ----------- ---------------------------- 12.8/44.0 MB 23.0 MB/s eta 0:00:02
   --------------- ------------------------ 17.6/44.0 MB 22.8 MB/s eta 0:00:02
   -------------------- ------------------- 22.8/44.0 MB 23.4 MB/s eta 0:00:01
   ------------------------- -------------- 27.8/44.0 MB 23.6 MB/s eta 0:00:01
   ----------------------------- ---------- 32.8/44.0 MB 23.4 MB/s eta 0:00:01
   -------------------------------- ------- 35.9/44.0 MB 22.5 MB/s eta 0:00:01
   ------------------------------------- -- 40.9/44.0 MB 22.7 MB/s eta 0:00:01
   ---------------------------------------  43.8/44.0 MB 22.8 MB/s eta 0:00:01
   ---------------------------------------- 44.0/44.0 MB 20.9 MB/s  0:00:02
   ---------------------------------------- 0.0/12.4 MB ? eta -:--


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import cv2
import numpy as np

# Load image
img = cv2.imread('C:\\Users\\VedantSonani\\OneDrive - inkeysolutions.com\\Desktop\\Extension\\images\\Product 1\\test.jpg')

# Apply Gaussian Blur to reduce noise
blurred = cv2.GaussianBlur(img, (21, 51), 3)

# Convert to grayscale and apply Otsu's thresholding
gray = cv2.cvtColor(blurred, cv2.COLOR_BGR2GRAY)
_, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Morphological operations to clean up the mask
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
mask = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=4)

# Apply mask to remove background
result = cv2.bitwise_and(img, img, mask=mask)

cv2.imwrite('C:\\Users\\VedantSonani\\OneDrive - inkeysolutions.com\\Desktop\\Extension\\images\\Product 1\\output.jpg', result)


True